# PixArt-alpha -- Fine-tuning DiT (Diffusion Transformer) + LoRA

Comparacion directa con `Difusores.ipynb`: misma tarea, mismo dataset, misma tecnica LoRA, arquitectura de denoiser completamente distinta.

## Que cambia respecto a Stable Diffusion?

SD 1.5 usa un **U-Net** como denoiser. PixArt-alpha usa un **DiT (Diffusion Transformer)**: el denoiser es un Transformer puro, sin convoluciones.

```
Imagen 512x512
  VAE encode (congelado)
Latente (4 x 64 x 64)
  Patchify (patch 2x2) -> 1024 tokens
  28 x DiT Block
    Self-Attention (imagen <-> imagen)
    Cross-Attention (imagen <- T5 texto)
    MLP Feed-Forward
  Un-patchify -> Ruido predicho
  DDPM denoising
Imagen 512x512
```

| Componente | SD 1.5 | PixArt-alpha |
|---|---|---|
| Denoiser | U-Net (conv) | DiT (Transformer puro) |
| Text encoder | CLIP (77t, 768D) | T5-XXL (120t, 4096D) |
| Params | ~860 M | ~611 M |

## Estrategia de memoria (T4 - 15.6 GB VRAM)

T5-XXL (~11 B params) se carga con device_map=auto, se pre-computan los 5000 embeddings, se guardan en disco y se descarga. El entrenamiento usa solo DiT + VAE (~3 GB VRAM).


## 0. Verificar GPU

In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU no encontrada — usando CPU.")
    print("El entrenamiento sera muy lento. Para usar GPU:")
    print("  1. Verifica que tienes una GPU Nvidia instalada")
    print("  2. Instala CUDA: https://developer.nvidia.com/cuda-downloads")
    print("  3. Instala PyTorch con CUDA: https://pytorch.org/get-started/locally/")

print(f"\nDevice: {device}")

GPU no encontrada — usando CPU.
El entrenamiento sera muy lento. Para usar GPU:
  1. Verifica que tienes una GPU Nvidia instalada
  2. Instala CUDA: https://developer.nvidia.com/cuda-downloads
  3. Instala PyTorch con CUDA: https://pytorch.org/get-started/locally/

Device: cpu


## 1. Instalar dependencias

In [ ]:
!pip install -q diffusers transformers peft datasets accelerate matplotlib torchvision sentencepiece protobuf tiktoken
!pip install -q bitsandbytes>=0.41.0
!pip install -q -U torchao
print("Dependencias listas")

## 2. Directorios de salida

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR  = "/content/drive/MyDrive/pixart_lora"   # modelo → Drive (persiste)
CACHE_DIR = "/content/emb_cache"                   # embeddings → disco local (más rápido)
os.makedirs(SAVE_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Modelo -> {SAVE_DIR}")
print(f"Cache  -> {CACHE_DIR}")

## 3. Imports y configuracion

In [ ]:
import random, gc, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from tqdm.notebook import tqdm

MODEL_ID    = "PixArt-alpha/PixArt-XL-2-512x512"
EPOCHS      = 3
LR          = 1e-4
BATCH_SIZE  = 2
GRAD_ACCUM  = 8
MAX_SAMPLES = 5000
IMAGE_SIZE  = 512
MAX_SEQ_LEN = 64    # Flickr8k captions ~10-20 palabras; 64 tokens cubre el 99%
LORA_RANK   = 8
LORA_ALPHA  = 32

TEST_PROMPTS = [
    "a dog running on a beach at sunset, photorealistic",
    "a cat sitting by a window in morning light",
    "two children playing in a park",
    "a mountain landscape with snow peaks",
    "a woman riding a horse in a field",
    "a bird flying over the ocean at dusk",
]
print(f"Modelo: {MODEL_ID} | Batch efectivo: {BATCH_SIZE*GRAD_ACCUM} | r={LORA_RANK}")

## 4. Fase 1 -- Pre-computar embeddings T5-XXL

T5-XXL (~11 B params, 4096D, 120 tokens) se carga una sola vez para codificar
los captions. Como esta siempre congelado, guardamos los embeddings y descargamos
el modelo para liberar VRAM al DiT.


In [ ]:
from transformers import T5EncoderModel, AutoTokenizer, BitsAndBytesConfig

# 8-bit: T5-XXL (~11 GB int8) cabe entero en la T4 (15.6 GB VRAM) → 0 RAM de CPU
bnb_8bit = BitsAndBytesConfig(load_in_8bit=True)
print("Cargando T5-XXL en 8-bit (~11 GB VRAM, sin tocar RAM)...")
tokenizer_t5 = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
t5_encoder   = T5EncoderModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder",
    quantization_config=bnb_8bit,
    device_map="auto", low_cpu_mem_usage=True,
)
t5_encoder.eval()
print(f"T5 cargado. VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Cargando Flickr8k...")
raw_ds   = load_dataset("jxie/flickr8k", split="train")
raw_ds   = raw_ds.select(range(min(MAX_SAMPLES, len(raw_ds))))
cap_cols = [c for c in raw_ds[0].keys() if c.startswith("caption")]
captions = [random.choice([raw_ds[i][c] for c in cap_cols]) for i in range(len(raw_ds))]
print(f"Muestras: {len(raw_ds)}")

# Streaming a disco con numpy memmap → 0 acumulación en RAM
n        = len(captions)
emb_gb   = n * MAX_SEQ_LEN * 4096 * 2 / 1e9
emb_path = f"{CACHE_DIR}/t5_embs.npy"
msk_path = f"{CACHE_DIR}/t5_masks.npy"
emb_mmap = np.memmap(emb_path, dtype='float16', mode='w+', shape=(n, MAX_SEQ_LEN, 4096))
msk_mmap = np.memmap(msk_path, dtype='int32',   mode='w+', shape=(n, MAX_SEQ_LEN))
print(f"Escribiendo {n} embeddings a disco ({emb_gb:.2f} GB float16)...")

ENCODE_BATCH = 4
with torch.no_grad():
    for i in tqdm(range(0, n, ENCODE_BATCH)):
        end  = min(i + ENCODE_BATCH, n)
        toks = tokenizer_t5(
            captions[i:end], padding="max_length", max_length=MAX_SEQ_LEN,
            truncation=True, return_tensors="pt"
        )
        out = t5_encoder(
            input_ids=toks.input_ids.to(t5_encoder.device),
            attention_mask=toks.attention_mask.to(t5_encoder.device)
        )
        emb_mmap[i:end] = out.last_hidden_state.cpu().half().numpy()
        msk_mmap[i:end] = toks.attention_mask.numpy().astype('int32')
        del out, toks

emb_mmap.flush(); msk_mmap.flush()
del emb_mmap, msk_mmap
print(f"Embeddings en disco: {emb_gb:.2f} GB")

del t5_encoder; gc.collect(); torch.cuda.empty_cache()
vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"T5 descargado. VRAM libre: {vram:.1f} GB")

## 5. Cargar DiT + VAE + aplicar LoRA

Con T5 descargado, cargamos solo el DiT y el VAE (~3 GB VRAM).
LoRA sobre to_q, to_k, to_v, to_out.0 en los 28 bloques DiT.


In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler
from diffusers.models import Transformer2DModel
from peft import LoraConfig, get_peft_model

transformer = Transformer2DModel.from_pretrained(
    MODEL_ID, subfolder="transformer", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float16)
vae.requires_grad_(False)
vae.to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.1, bias="none",
)
transformer = get_peft_model(transformer, lora_config)
transformer.print_trainable_parameters()
transformer.to(device)
print(f"VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 6. Dataset

In [ ]:
class PixArtDataset(Dataset):
    def __init__(self, hf_ds, embeddings, masks, transform):
        self.ds=hf_ds; self.embeddings=embeddings
        self.masks=masks; self.transform=transform
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        return {"pixel_values": self.transform(self.ds[idx]["image"].convert("RGB")),
                "encoder_hidden_states": self.embeddings[idx],
                "encoder_attention_mask": self.masks[idx]}

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

n = len(raw_ds)
t5_embs  = torch.from_numpy(
    np.memmap(f"{CACHE_DIR}/t5_embs.npy", dtype='float16', mode='r',
              shape=(n, MAX_SEQ_LEN, 4096)).copy()
)
t5_masks = torch.from_numpy(
    np.memmap(f"{CACHE_DIR}/t5_masks.npy", dtype='int32', mode='r',
              shape=(n, MAX_SEQ_LEN)).copy()
)
print(f"Embeddings en RAM: {t5_embs.nbytes/1e9:.2f} GB (float16)")

dataset    = PixArtDataset(raw_ds, t5_embs, t5_masks, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)
print(f"Dataset: {len(dataset)} muestras | {len(dataloader)} batches/ep")

## 7. Entrenamiento LoRA sobre DiT

Loop identico al de Difusores.ipynb (DDPM + MSE loss), pero el denoiser es el DiT:
```
latente -> add_noise(t) -> DiT+LoRA(noisy, t, T5_emb) -> ruido_predicho
loss = MSE(ruido_predicho, ruido_real)
```
Tiempo estimado T4: ~60-80 min.


In [ ]:
optimizer = torch.optim.AdamW(transformer.parameters(), lr=LR, weight_decay=1e-2)
scaler    = torch.amp.GradScaler("cuda")
transformer.train()

for epoch in range(EPOCHS):
    print(f"\n-- Epoca {epoch+1}/{EPOCHS} --")
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(dataloader)):
        pv     = batch["pixel_values"].to(device, dtype=torch.float16)
        enc_hs = batch["encoder_hidden_states"].to(device, dtype=torch.float16)
        enc_am = batch["encoder_attention_mask"].to(device)
        with torch.no_grad():
            latents = vae.encode(pv).latent_dist.sample() * vae.config.scaling_factor
        noise     = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device=device).long()
        noisy     = noise_scheduler.add_noise(latents, noise, timesteps)
        with torch.amp.autocast("cuda"):
            pred = transformer(hidden_states=noisy, timestep=timesteps,
                               encoder_hidden_states=enc_hs,
                               encoder_attention_mask=enc_am, return_dict=False)[0]
            loss = F.mse_loss(pred.float(), noise.float())
        scaler.scale(loss / GRAD_ACCUM).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total_loss += loss.item()
    print(f"Loss promedio: {total_loss / len(dataloader):.4f}")

print("Entrenamiento terminado")

## 8. Guardar pesos LoRA

In [ ]:
transformer.save_pretrained(f"{SAVE_DIR}/lora_transformer")
config = {"model_id": MODEL_ID, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
          "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
          "grad_accum": GRAD_ACCUM, "max_samples": MAX_SAMPLES}
with open(f"{SAVE_DIR}/config.json", "w") as f: json.dump(config, f, indent=2)
print(f"Guardado en {SAVE_DIR}")

## 9. Inferencia

enable_sequential_cpu_offload() mueve T5 a CPU cuando no se usa, evitando OOM en T4.


In [ ]:
from diffusers import PixArtAlphaPipeline

transformer.eval()
pipe = PixArtAlphaPipeline.from_pretrained(
    MODEL_ID, transformer=transformer, torch_dtype=torch.float16)
pipe.enable_sequential_cpu_offload()

@torch.no_grad()
def generate(prompt, steps=20, guidance=4.5, seed=42):
    return pipe(prompt, num_inference_steps=steps, guidance_scale=guidance,
                generator=torch.Generator().manual_seed(seed)).images[0]

img = generate(TEST_PROMPTS[0])
plt.figure(figsize=(6,6)); plt.imshow(img); plt.axis("off")
plt.title("PixArt-alpha DiT + LoRA (fine-tuned Flickr8k)", fontsize=11)
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/sample.png", dpi=120); plt.show()

## 10. Cuadricula de muestras

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, p in zip(axes.flat, TEST_PROMPTS):
    print(f"  {p[:55]}")
    ax.imshow(generate(p)); ax.axis("off"); ax.set_title(p[:55], fontsize=8)
plt.suptitle("PixArt-alpha DiT + LoRA -- Flickr8k 512x512", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/grid.png", dpi=100); plt.show()

## 11. Comparacion final vs SD + LoRA

| | SD 1.5 + LoRA | PixArt-alpha DiT + LoRA |
|---|---|---|
| Denoiser | U-Net (conv) | DiT (Transformer puro) |
| Text encoder | CLIP-L (77t, 768D) | T5-XXL (120t, 4096D) |
| Params totales | ~860 M | ~611 M |
| Params LoRA | ~797 K (0.09%) | ~720 K (0.12%) |
| Estrategia memoria | CLIP en GPU | Pre-compute T5 + descargar |
| Inferencia T4 | ~5 s (30 pasos) | ~8-12 s (20 pasos) |

El cambio U-Net -> DiT elimina las convoluciones del denoiser.
Combinado con T5-XXL, PixArt-alpha logra mejor alineamiento texto-imagen
en prompts complejos y descripciones detalladas.
